[Back to Machine Learning guideline](Machine Learning.html)


## **Geometric & Kernel Methods**
### **Nearest neighbour algorithm**
#### **Distance Measures**

- Euclidean Distance: $d(x, y) = \sqrt{\sum (x_i - y_i)^2}$, used for continous numerical features.
- Mahattan Distance: $d(x, y) = \sum |x_i - y_i|$, more robust to outlier than Euclidean distance.
- Minkowski Distance: $d(x, y) = \left( \sum |x_i - y_i|^p \right)^{1/p}$, generalization of Euclidean (p=2) and Manhattan (p=1).
- Cosine Similarity: $\cos(\theta) = \frac{x \cdot y}{\|x\| \|y\|}$, measures similarity in direction, often used in high-dimensional spaces such as text data.
#### **1-Nearest Neighbour**(1-NN)
1-NN is the simplest form of nearest neighbour algorithms. It classifies a new data point by assigning it the label of its single closest training example based on a chosen distance metric. Advantages: Simple and easy. No training phase(lazy learing). Can model complex and nonlinear decision boundaries. Disadvantages: Highly sensitive to noise and outliers. Can easily overfit. Performance degrades in high-dimensional spaces.

##### **Pseudocode**:

```
Input:
    Training set D = {(x1, y1), (x2, y2), ..., (xn, yn)}
    Query point x
    Distance function d(·, ·)

Algorithm:
1. Initialize min_distance = ∞
2. Initialize nearest_label = None

3. For each (xi, yi) in D:
       compute distance = d(x, xi)

       If distance < min_distance:
            min_distance = distance
            nearest_label = yi

4. Return nearest_label
```

#### **K-Nearest Neighour**(KNN)
KNN is an extension of 1-NN that classifies a new data point based on the labels of its k closest neighbours in the training set, typically using majority voting for classification. Compared to 1-NN, KNN is more robust to noise since it considers multiple neighbours rather than relying on a single point. Its advantages include simplicity, flexibility in capturing nonlinear decision boundaries, and no requirement for explicit model training. However, it is sensitive to the choice of k, computationally expensive at prediction time, and suffers from the curse of dimensionality in high-dimensional data. KNN is most suitable for small to medium-sized datasets, problems with irregular decision boundaries, and scenarios where local similarity is meaningful, such as recommendation systems, pattern recognition, and anomaly detection.

In KNN, ties may occur when multiple classes receive the same number of votes. Common tie-breaking strategies include selecting the class with the nearest neighbour among the tied classes, using distance-weighted voting to avoid ties, choosing an odd value of k to reduce the chance of ties, or randomly selecting a class as a last resort.

##### **Pseudocode**:
```
Input:
- Training set D = {(x1, y1), (x2, y2), ..., (xn, yn)}
- Query point x
- Number of neighbours k
- Distance function d(·, ·)

Algorithm:
1. For each training example (xi, yi) in D:
      compute distance di = d(x, xi)
2. Sort all training examples by distance di in ascending order
3. Select the k nearest examples
4. Count the votes for each class among the k neighbours
5. Return the class with the highest number of votes

Output:
- Predicted class label
```

<details>
<summary>Python Implementation</summary>

```python
import numpy as np
from collections import Counter

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict_one(self, x):
        distances = []

        for i, x_train in enumerate(self.X_train):
            distance = np.sqrt(np.sum((x - x_train) ** 2))
            distances.append((distance, self.y_train[i]))

        distances.sort(key=lambda item: item[0])
        k_nearest = distances[:self.k]

        labels = [label for _, label in k_nearest]
        return Counter(labels).most_common(1)[0][0]

    def predict(self, X):
        X = np.array(X)
        return [self.predict_one(x) for x in X]

X_train = [
    [1, 1],
    [2, 2],
    [3, 3],
    [8, 8],
    [9, 9],
    [10, 10]
]

y_train = ["A", "A", "A", "B", "B", "B"]

X_test = [
    [2.5, 2.5],
    [8.5, 8.5]
]

model = KNN(k=3)
model.fit(X_train, y_train)

print(model.predict(X_test))
```
</details>

<details>
<summary>Python sklearn implementation</summary>

```python
from sklearn.neighbors import KNeighborsClassifier

X_train = [
    [1, 1],
    [2, 2],
    [3, 3],
    [8, 8],
    [9, 9],
    [10, 10]
]

y_train = ["A", "A", "A", "B", "B", "B"]

X_test = [
    [2.5, 2.5],
    [8.5, 8.5]
]

model = KNeighborsClassifier(
    n_neighbors=3,
    weights="uniform",
    metric="euclidean"
)

model.fit(X_train, y_train)

print(model.predict(X_test))
```
</details>

#### **Weighted Nearest Neighbours**
Weighted Nearest Neighbours is an extension of KNN in which each of the k nearest neighbours contributes to the prediction with a weight based on its distance to the query point, so that closer neighbours have a stronger influence than farther ones. Instead of simple majority voting, the model computes a weighted vote, typically using functions such as inverse distance (e.g., $( w_i = \frac{1}{d(x, x_i)})$. This approach reduces the impact of less relevant distant neighbours and helps resolve tie situations naturally. Its advantages include improved robustness and often better accuracy compared to standard KNN, especially when nearby points are more informative. However, it still suffers from high computational cost at prediction time, requires careful choice of weighting function, and remains sensitive to feature scaling and high-dimensional data. Weighted nearest neighbours are most suitable in scenarios where local proximity is highly meaningful and closer observations should dominate the decision, such as recommendation systems, spatial data analysis, and pattern recognition tasks.

##### **Pseudocode**:
```
Input:
- Training set D = {(x1, y1), (x2, y2), ..., (xn, yn)}
- Query point x
- Number of neighbours k
- Distance function d(·, ·)

Algorithm:
1. For each training example (xi, yi) in D:
      compute distance di = d(x, xi)
2. Sort all training examples by distance di in ascending order
3. Select the k nearest examples
4. For each selected neighbour:
      compute weight wi = 1 / (di + ε) # ε is a very small number, used to avoid dividing by 0 when the distance is 0
5. For each class:
      sum the weights of neighbours belonging to that class
6. Return the class with the highest total weight

Output:
- Predicted class label
```


<details>
<summary>Python implementation</summary>

```python
import numpy as np
from collections import defaultdict

class WeightedKNN:
    def __init__(self, k=3, epsilon=1e-8):
        self.k = k
        self.epsilon = epsilon

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict_one(self, x):
        distances = []

        for i, x_train in enumerate(self.X_train):
            distance = np.sqrt(np.sum((x - x_train) ** 2))
            distances.append((distance, self.y_train[i]))

        distances.sort(key=lambda item: item[0])
        k_nearest = distances[:self.k]

        class_weights = defaultdict(float)

        for distance, label in k_nearest:
            weight = 1 / (distance + self.epsilon)
            class_weights[label] += weight

        return max(class_weights, key=class_weights.get)

    def predict(self, X):
        X = np.array(X)
        return [self.predict_one(x) for x in X]

X_train = [
    [1, 1],
    [2, 2],
    [3, 3],
    [8, 8],
    [9, 9],
    [10, 10]
]

y_train = ["A", "A", "A", "B", "B", "B"]

X_test = [
    [2.5, 2.5],
    [8.5, 8.5]
]

model = WeightedKNN(k=3)
model.fit(X_train, y_train)

print(model.predict(X_test))
```
</details>

<details>
<summary>Python sklearn implementation</summary>

```python
from sklearn.neighbors import KNeighborsClassifier

X_train = [
    [1, 1],
    [2, 2],
    [3, 3],
    [8, 8],
    [9, 9],
    [10, 10]
]

y_train = ["A", "A", "A", "B", "B", "B"]

X_test = [
    [2.5, 2.5],
    [8.5, 8.5]
]

model = KNeighborsClassifier(
    n_neighbors=3,
    weights="distance",
    metric="euclidean"
)

model.fit(X_train, y_train)

print(model.predict(X_test))
```
</details>